# Route XML Filter
Objective: Filter the complete Bus Routes of Berlin accoarding to a csv containing the desired routes \
Author: Sven Vorgheim \
Disclaimer: The creation of this file was aided by ChatGPT

In [1]:
# Imports
import pandas as pd
from lxml import etree
import math
import os

Assumes the use of "berlin_bus_rou.xml" as given in the BeST-Scenario \
Requieres a "depot_line_type.csv" containing a coloumn "Line" with the desired Lines as string

In [4]:
# Read the csv, select coloumn
specified_routes = pd.read_csv("./depot_line_type.csv",sep=",")
specified = set(specified_routes["Line"])
# Parse xml file
tree = etree.parse("../../sumo/berlin_bus.rou.xml")
root = tree.getroot()
# xml anchor contains line as the first part of the anchor
# removes any undesired routes and flows
for route in root.xpath("./route"):
    anchor = route.get("id").split("_", 1)[0]
    if anchor not in specified:
        root.remove(route)

for flow in root.xpath("./flow"):
    anchor = flow.get("id").split("_", 1)[0]
    if anchor not in specified:
        root.remove(flow)

# write result to file as intermediated chache
tree.write(
    "filtered_flows_routes.xml",
    pretty_print=True,
    xml_declaration=True,
    encoding="UTF-8",
)

Calculate additonal information from routes and flows \
Number of simultaneous busses is calculated from route duration and flow period \
nr_of_buses = route.duration/flow.period \
nr_of_repetitions = (flow_end-flow_begin)/duration \
Each Bus on the line must depart $n*period$ after n =1 

Define deadheading routes by Adding departure and arrival areas? Sumo Taz

In [5]:
# Helper Function time in seconds from midnight
def parse_time(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

In [6]:
tree = etree.parse("./filtered_flows_routes.xml")
root = tree.getroot()

rows = []

for route in root.xpath("./route"):
    anchor = route.get("id")
    for flow in root.xpath("./flow"):
        if flow.get("route") == anchor:
            period = float(flow.get("period"))
            duration = float(route.findall("stop")[-1].get("until"))
            nr_of_buses = math.ceil(int(duration) / int(period))
            flow_end = parse_time(flow.get("end"))
            flow_begin = parse_time(flow.get("begin"))
            # print(flow.get("begin"),flow_begin,flow.get("end"),flow_end)
            nr_of_repetitions = (flow_end-flow_begin)/duration
            rows.append({
                "route": anchor,
                "flow_end": flow_end,
                "flow_begin": flow_begin,
                "period": period,
                "duration": duration,
                "nr_of_buses": nr_of_buses,
                "nr_of_repetitions": int(nr_of_repetitions),
            })
            root.remove(flow)

# write final result to file
tree.write(
    "filtered_routes.xml",
    pretty_print=True,
    xml_declaration=True,
    encoding="UTF-8",
)
os.remove("./filtered_flows_routes.xml")

route_calculations = pd.DataFrame(rows)
print(route_calculations)
# Total result of simultaneous buses
print(sum(route_calculations["nr_of_buses"]))
route_calculations.to_csv("route_calculations.csv", index=False)

        route  flow_end  flow_begin  period  duration  nr_of_buses  \
0    101_9633     85800       15120   600.0    3860.0            7   
1    101_9646     85740        1740   600.0    3980.0            7   
2    109_9722     86160       19860   720.0    1820.0            3   
3    109_9724     86040       19200   600.0    2000.0            4   
4    110_9735     85800       18540  1200.0    1970.0            2   
..        ...       ...         ...     ...       ...          ...   
76  X83_12561     85560       15960   540.0    2900.0            6   
77  X83_12570     85380        1200   540.0    3140.0            6   
78  143_12751     86040        1800  1200.0    2540.0            3   
79  M43_12763     85260        1260   480.0    3200.0            7   
80  M43_12771     85620        1860   360.0    3200.0            9   

    nr_of_repetitions  
0                  18  
1                  21  
2                  36  
3                  33  
4                  34  
..             

Created merged csv containing information on Lines

In [7]:
route_df = pd.read_csv("./route_calculations.csv")
line_df = pd.read_csv("./depot_line_type.csv")

# Extract the line name from the route ID
route_df["Line"] = route_df["route"].str.split("_").str[0]

# Join on the Line column
merged = route_df.merge(line_df, on="Line", how="left")

merged.to_csv("merged_routes.csv", index=False)
os.remove("./route_calculations.csv")
print(merged.head())

      route  flow_end  flow_begin  period  duration  nr_of_buses  \
0  101_9633     85800       15120   600.0    3860.0            7   
1  101_9646     85740        1740   600.0    3980.0            7   
2  109_9722     86160       19860   720.0    1820.0            3   
3  109_9724     86040       19200   600.0    2000.0            4   
4  110_9735     85800       18540  1200.0    1970.0            2   

   nr_of_repetitions Line         Depot Type  Both Depots  DoubleDecker  
0                 18  101  Cicerostraße   DL            0             0  
1                 21  101  Cicerostraße   DL            0             0  
2                 36  109  Müllerstraße   EN            0             0  
3                 33  109  Müllerstraße   EN            0             0  
4                 34  110  Cicerostraße   EN            0             0  
